# Aprendizaje por Refuerzo: Q-learning vs SARSA en Taxi
**Entorno:** Taxi (Gymnasium) · **Algoritmos:** Q-learning y SARSA tabulares

Este cuaderno contiene todo el flujo: definición del entorno, implementación de los agentes,
experimentos con varias semillas, análisis de hiperparámetros, comparación con la política
óptima (iteración de valor) y gráficas. Todas las figuras se guardan en la carpeta `resultados/`.

In [ ]:
# !pip install -q gymnasium numpy matplotlib      # (descomentar en Google Colab)

import json, os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

warnings.filterwarnings("ignore")
OUT = "resultados"
os.makedirs(OUT, exist_ok=True)

# Estilo de gráficas
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
COL = {"Q-learning": "#1f77b4", "SARSA": "#d62728", "Aleatorio": "#7f7f7f", "Óptima": "#2ca02c"}

## 1. Entorno
Se usa `Taxi-v4` (en versiones recientes de Gymnasium `Taxi-v3` está obsoleto).
* **Estados:** 500 = 25 posiciones del taxi × 5 ubicaciones del pasajero × 4 destinos.
* **Acciones:** 0 sur, 1 norte, 2 este, 3 oeste, 4 recoger, 5 dejar.
* **Recompensas:** −1 por paso, +20 al entregar, −10 por recoger/dejar de forma ilegal.

In [ ]:
try:
    env = gym.make("Taxi-v4")
except Exception:
    env = gym.make("Taxi-v3")

N_S, N_A = env.observation_space.n, env.action_space.n
print(f"Estados: {N_S} | Acciones: {N_A}")

## 2. Agentes (Q-learning y SARSA)
Ambos usan política ε-greedy con decaimiento exponencial de ε.
* **Q-learning (off-policy):** `Q(s,a) += α [ r + γ·max_a' Q(s',a') − Q(s,a) ]`
* **SARSA (on-policy):** `Q(s,a) += α [ r + γ·Q(s',a') − Q(s,a) ]`, con `a'` la acción realmente elegida.

In [ ]:
def elegir_accion(Q, s, eps, rng):
    """Política ε-greedy: explora con prob. ε, si no explota la mejor acción conocida."""
    if rng.random() < eps:
        return int(rng.integers(N_A))
    return int(np.argmax(Q[s]))


def entrenar(algoritmo="q", alpha=0.1, gamma=0.99, eps_decay=0.999, eps_min=0.01,
             episodios=5000, semilla=0):
    """Entrena un agente tabular. Devuelve la tabla Q, retornos, pasos y trayectoria de ε."""
    rng = np.random.default_rng(semilla)
    Q = np.zeros((N_S, N_A))
    eps = 1.0
    retornos, pasos, epsilons = [], [], []

    for ep in range(episodios):
        s, _ = env.reset(seed=int(rng.integers(1_000_000_000)))
        a = elegir_accion(Q, s, eps, rng)
        terminado, G, t = False, 0.0, 0

        while not terminado:
            s2, r, term, trunc, _ = env.step(a)
            terminado = term or trunc
            a2 = elegir_accion(Q, s2, eps, rng)       # acción siguiente (la usa SARSA)

            if term:                                   # estado terminal: sin valor futuro
                objetivo = r
            elif algoritmo == "q":
                objetivo = r + gamma * np.max(Q[s2])
            else:
                objetivo = r + gamma * Q[s2, a2]

            Q[s, a] += alpha * (objetivo - Q[s, a])    # actualización TD
            s, a = s2, a2
            G += r; t += 1

        epsilons.append(eps)
        eps = max(eps_min, eps * eps_decay)            # decaimiento de la exploración
        retornos.append(G); pasos.append(t)

    return Q, np.array(retornos), np.array(pasos), np.array(epsilons)


def evaluar(politica, n=500, semilla_base=10_000):
    """Evalúa una política greedy (sin exploración). `politica` es una función s -> a."""
    R, T, ok = [], [], 0
    for i in range(n):
        s, _ = env.reset(seed=semilla_base + i)
        terminado, G, t, r = False, 0.0, 0, 0
        while not terminado:
            s, r, term, trunc, _ = env.step(int(politica(s)))
            G += r; t += 1; terminado = term or trunc
        R.append(G); T.append(t); ok += int(term and r == 20)
    return {"retorno": float(np.mean(R)), "pasos": float(np.mean(T)), "exito": ok / n,
            "retornos": R}


def media_movil(x, k=100):
    return np.convolve(x, np.ones(k) / k, mode="valid")

## 3. Referencias: política aleatoria y política óptima
La política óptima se calcula con **iteración de valor** usando el modelo `env.unwrapped.P`.
Sirve como "techo" para saber qué tan cerca llegan los agentes.

In [ ]:
def iteracion_valor(gamma=0.99, tol=1e-10):
    P = env.unwrapped.P
    V = np.zeros(N_S)
    while True:
        delta = 0
        for s in range(N_S):
            q = [sum(p * (r + gamma * V[s2] * (not d)) for p, s2, r, d in P[s][a])
                 for a in range(N_A)]
            nuevo = max(q); delta = max(delta, abs(nuevo - V[s])); V[s] = nuevo
        if delta < tol:
            break
    Qopt = np.array([[sum(p * (r + gamma * V[s2] * (not d)) for p, s2, r, d in P[s][a])
                      for a in range(N_A)] for s in range(N_S)])
    return Qopt

Q_opt = iteracion_valor()
res_opt = evaluar(lambda s: np.argmax(Q_opt[s]), n=300)   # mismos 300 episodios que los agentes
rng_az = np.random.default_rng(123)
res_azar = evaluar(lambda s: rng_az.integers(N_A), n=200)
print("Aleatorio:", {k: round(v, 2) for k, v in res_azar.items() if k != "retornos"})
print("Óptima   :", {k: round(v, 2) for k, v in res_opt.items() if k != "retornos"})

## 4. Experimento principal: Q-learning vs SARSA (5 semillas × 5000 episodios)

In [ ]:
SEMILLAS = [0, 1, 2, 3, 4]
EPISODIOS = 5000
datos = {"Q-learning": {"ret": [], "pasos": [], "Q": []}, "SARSA": {"ret": [], "pasos": [], "Q": []}}
eps_tray = None
tiempos = {}

for nombre, alg in [("Q-learning", "q"), ("SARSA", "sarsa")]:
    t0 = time.time()
    for sem in SEMILLAS:
        Q, ret, pas, eps_tray = entrenar(alg, episodios=EPISODIOS, semilla=sem)
        datos[nombre]["ret"].append(ret); datos[nombre]["pasos"].append(pas); datos[nombre]["Q"].append(Q)
    tiempos[nombre] = (time.time() - t0) / len(SEMILLAS)
    print(f"{nombre}: entrenamiento medio {tiempos[nombre]:.1f}s por semilla")

# Evaluación greedy de cada semilla
evals = {n: [evaluar(lambda s, Q=Q: np.argmax(Q[s]), n=300) for Q in datos[n]["Q"]] for n in datos}
resumen = {}
for n in evals:
    resumen[n] = {m: float(np.mean([e[m] for e in evals[n]])) for m in ["retorno", "pasos", "exito"]}
    resumen[n]["retorno_std"] = float(np.std([e["retorno"] for e in evals[n]]))
    print(n, {k: round(v, 3) for k, v in resumen[n].items()})

### Figura 1 – Curvas de aprendizaje (media ± desviación entre semillas)

In [ ]:
def curva(n, clave):
    M = np.array([media_movil(x) for x in datos[n][clave]])
    return M.mean(0), M.std(0)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for n in datos:
    m, s = curva(n, "ret"); x = np.arange(len(m))
    ax[0].plot(x, m, color=COL[n], label=n); ax[0].fill_between(x, m - s, m + s, color=COL[n], alpha=0.2)
    m, s = curva(n, "pasos")
    ax[1].plot(x, m, color=COL[n], label=n); ax[1].fill_between(x, m - s, m + s, color=COL[n], alpha=0.2)
ax[0].axhline(res_opt["retorno"], color=COL["Óptima"], ls="--", label="Política óptima")
ax[0].set_ylim(-800, 30); ax[0].set_title("Retorno por episodio (media móvil de 100)")
ax[0].set_xlabel("Episodio"); ax[0].set_ylabel("Retorno"); ax[0].legend()
ax[1].axhline(res_opt["pasos"], color=COL["Óptima"], ls="--", label="Política óptima")
ax[1].set_ylim(0, 200); ax[1].set_title("Pasos por episodio (media móvil de 100)")
ax[1].set_xlabel("Episodio"); ax[1].set_ylabel("Pasos"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_curvas_aprendizaje.png", dpi=150); plt.show()

### Figura 2 – Zoom a la fase de convergencia y decaimiento de ε

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for n in datos:
    m, s = curva(n, "ret"); x = np.arange(len(m))
    ax[0].plot(x, m, color=COL[n], label=n); ax[0].fill_between(x, m - s, m + s, color=COL[n], alpha=0.2)
ax[0].axhline(res_opt["retorno"], color=COL["Óptima"], ls="--", label="Política óptima")
ax[0].set_xlim(1500, EPISODIOS - 100); ax[0].set_ylim(-30, 12)
ax[0].set_title("Zoom: fase de convergencia"); ax[0].set_xlabel("Episodio"); ax[0].set_ylabel("Retorno"); ax[0].legend()
ax[1].plot(eps_tray, color="#9467bd"); ax[1].set_title("Decaimiento de la exploración ε")
ax[1].set_xlabel("Episodio"); ax[1].set_ylabel("ε")
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_zoom_epsilon.png", dpi=150); plt.show()

### Figura 3 – Comparación final: aleatorio vs Q-learning vs SARSA vs óptima

In [ ]:
nombres = ["Aleatorio", "Q-learning", "SARSA", "Óptima"]
ret_v = [res_azar["retorno"], resumen["Q-learning"]["retorno"], resumen["SARSA"]["retorno"], res_opt["retorno"]]
pas_v = [res_azar["pasos"], resumen["Q-learning"]["pasos"], resumen["SARSA"]["pasos"], res_opt["pasos"]]
exi_v = [res_azar["exito"] * 100, resumen["Q-learning"]["exito"] * 100, resumen["SARSA"]["exito"] * 100, res_opt["exito"] * 100]

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for a, vals, tit in zip(ax, [ret_v, pas_v, exi_v], ["Retorno medio", "Pasos medios", "Tasa de éxito (%)"]):
    barras = a.bar(nombres, vals, color=[COL[n] for n in nombres])
    a.set_title(tit); a.tick_params(axis="x", rotation=20)
    for b, v in zip(barras, vals):
        a.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}", ha="center",
               va="bottom" if v >= 0 else "top", fontsize=9)
ax[0].set_ylim(min(ret_v) * 1.15, 40)
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_comparacion_final.png", dpi=150); plt.show()

## 5. Sensibilidad a hiperparámetros (Q-learning, 3 semillas × 3000 episodios)
Se varía un hiperparámetro a la vez, dejando los demás en sus valores base
(α = 0.1, γ = 0.99, decaimiento de ε = 0.999).

In [ ]:
BASE = dict(alpha=0.1, gamma=0.99, eps_decay=0.999)
GRID = {"alpha": [0.01, 0.05, 0.1, 0.3, 0.5, 0.9],
        "gamma": [0.5, 0.8, 0.9, 0.95, 0.99],
        "eps_decay": [0.99, 0.995, 0.999, 0.9995, 0.9999]}
sens = {}
for hp, valores in GRID.items():
    sens[hp] = []
    for v in valores:
        cfg = {**BASE, hp: v}
        rets = []
        for sem in [0, 1, 2]:
            Q, *_ = entrenar("q", episodios=3000, semilla=sem, **cfg)
            rets.append(evaluar(lambda s, Q=Q: np.argmax(Q[s]), n=200)["retorno"])
        sens[hp].append((v, float(np.mean(rets)), float(np.std(rets))))
        print(hp, v, round(np.mean(rets), 2))

### Figura 4 – Sensibilidad a α, γ y decaimiento de ε

In [ ]:
titulos = {"alpha": "Tasa de aprendizaje α", "gamma": "Factor de descuento γ", "eps_decay": "Decaimiento de ε"}
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for a, hp in zip(ax, GRID):
    v, m, s = zip(*sens[hp]); xs = range(len(v))
    a.bar(xs, m, yerr=s, capsize=4, color="#1f77b4", alpha=0.85)
    a.axhline(res_opt["retorno"], color=COL["Óptima"], ls="--", label="Óptima")
    a.set_xticks(list(xs)); a.set_xticklabels([str(x) for x in v]); a.set_title(titulos[hp])
    a.set_xlabel("Valor"); a.set_ylabel("Retorno de evaluación"); a.set_ylim(max(min(m) - 30, -250), 20)
ax[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig4_sensibilidad.png", dpi=150); plt.show()

## 6. Política aprendida (visualización)
Para un caso concreto (pasajero en R, destino en B) se dibuja la mejor acción en cada celda,
primero con el taxi vacío (va a recoger) y luego con el pasajero dentro (va a dejarlo).

In [ ]:
LUGARES = {0: (0, 0, "R"), 1: (0, 4, "G"), 2: (4, 0, "Y"), 3: (4, 3, "B")}
FLECHAS = {0: (0, 0.3), 1: (0, -0.3), 2: (0.3, 0), 3: (-0.3, 0)}   # (dx, dy); el eje y está invertido: sur = abajo
Q_vis = datos["Q-learning"]["Q"][0]
PASAJERO, DESTINO = 0, 3

def dibujar_politica(ax, pas_loc, titulo):
    ax.set_xlim(-0.5, 4.5); ax.set_ylim(4.5, -0.5); ax.set_aspect("equal"); ax.grid(False)
    for i in range(6):
        ax.axhline(i - 0.5, color="#bbb", lw=0.8); ax.axvline(i - 0.5, color="#bbb", lw=0.8)
    for fila in range(5):
        for col in range(5):
            s = env.unwrapped.encode(fila, col, pas_loc, DESTINO)
            a = int(np.argmax(Q_vis[s]))
            if a in FLECHAS:
                dx, dy = FLECHAS[a]
                ax.arrow(col - dx, fila - dy, 2 * dx, 2 * dy, head_width=0.13, color="k", length_includes_head=True)
            else:
                ax.text(col, fila, "P" if a == 4 else "D", ha="center", va="center",
                        fontsize=15, fontweight="bold", color="#d62728")
    for k, (f, c, n) in LUGARES.items():
        ax.text(c - 0.4, f - 0.35, n, fontsize=11, color="#1f77b4", fontweight="bold")
    ax.set_title(titulo); ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
dibujar_politica(ax[0], PASAJERO, "Pasajero en R, destino B\nTaxi vacío: ir a recoger (P)")
dibujar_politica(ax[1], 4, "Pasajero dentro del taxi, destino B\nIr a dejar (D)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig5_politica.png", dpi=150); plt.show()

### Figura 6 – Distribución de retornos en evaluación y valor máximo de Q por estado

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for n in ["Q-learning", "SARSA"]:
    todos = np.concatenate([e["retornos"] for e in evals[n]])
    ax[0].hist(todos, bins=np.arange(-10, 22, 1), alpha=0.6, color=COL[n], label=n)
ax[0].set_title("Distribución del retorno en evaluación greedy")
ax[0].set_xlabel("Retorno por episodio"); ax[0].set_ylabel("Frecuencia"); ax[0].legend()

mapa = np.zeros((5, 5))
for f in range(5):
    for c in range(5):
        mapa[f, c] = Q_vis[env.unwrapped.encode(f, c, 4, DESTINO)].max()   # pasajero dentro, destino B
im = ax[1].imshow(mapa, cmap="viridis"); ax[1].grid(False)
for f in range(5):
    for c in range(5):
        ax[1].text(c, f, f"{mapa[f, c]:.1f}", ha="center", va="center", color="w", fontsize=9)
ax[1].set_title("max Q(s,a) con pasajero a bordo (destino B)"); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.savefig(f"{OUT}/fig6_distribucion_valores.png", dpi=150); plt.show()

## 7. Métricas de convergencia y guardado de resultados

In [ ]:
def episodio_convergencia(n, umbral=0.0):
    """Primer episodio en que la media móvil (100) supera el umbral, promediado entre semillas."""
    eps_ = []
    for ret in datos[n]["ret"]:
        mm = media_movil(ret)
        idx = np.argmax(mm > umbral)
        eps_.append(int(idx) if mm[idx] > umbral else EPISODIOS)
    return float(np.mean(eps_)), float(np.std(eps_))

conv = {n: episodio_convergencia(n) for n in datos}
ultimos = {n: float(np.mean([r[-500:].mean() for r in datos[n]["ret"]])) for n in datos}
print("Episodios hasta retorno medio > 0:", conv)
print("Retorno de entrenamiento (últimos 500 ep.):", ultimos)

resultados = {"aleatorio": {k: v for k, v in res_azar.items() if k != "retornos"},
              "optima": {k: v for k, v in res_opt.items() if k != "retornos"},
              "resumen": resumen, "convergencia": conv, "entrenamiento_final": ultimos,
              "tiempos_s": tiempos, "sensibilidad": sens}
with open(f"{OUT}/resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print("Resultados y figuras guardados en", OUT)